In [1]:
import sys
from pathlib import Path

# notebook is inside: RAG/notebooks
# project root is one level up
PROJECT_ROOT = Path.cwd().parent

# make src/ importable
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())



PROJECT_ROOT: C:\Users\Vohita\RAG
src exists: True


In [2]:
DATA_PATH = Path.cwd().parent / "data_legal"

In [4]:
from src.retrieval.bm25 import BM25Index
from src.retrieval.faiss import FAISSIndex
# from src.storage.metadata import MetadataStore
# from src.retrieval.hybridretriver import HybridRetriever
# from src.security.user_context import UserContext

In [5]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path.cwd().parent / "data_legal"

chunks = pd.read_csv(DATA_PATH / "rag_corpus_chunks.csv")

# make sure row_id exists
chunks = chunks.reset_index(drop=True)
chunks["row_id"] = chunks.index

print("Chunks loaded:", chunks.shape)
chunks.head()



Chunks loaded: (458, 7)


,chunk_id,doc_id,domain,chunk_index,estimated_tokens,chunk_text,row_id
0,LC00001,LEGAL_DOC_001,legal,0,51,Company shall not specify the business practic...,0
1,LC00002,LEGAL_DOC_001,legal,0,308,In the event that Licensor grants to another V...,1
2,LC00003,LEGAL_DOC_001,legal,0,43,During the License Term (which is identified i...,2
3,LC00004,LEGAL_DOC_001,legal,0,90,"""Subject to Licensee's on\xadgoing compliance ...",3
4,LC00005,LEGAL_DOC_001,legal,0,122,"During the Term, (a) Women.com will not buy, s...",4


In [7]:
chunks.shape


(458, 7)

In [8]:
chunks.columns

Index(['chunk_id', 'doc_id', 'domain', 'chunk_index', 'estimated_tokens',
       'chunk_text', 'row_id'],
      dtype='object')

In [7]:
# retrievals.columns

Index(['run_id', 'example_id', 'scenario_id', 'query_id', 'split',
       'query_domain', 'difficulty', 'retrieval_strategy', 'rank', 'chunk_id',
       'retrieval_score', 'is_relevant'],
      dtype='object')

In [8]:
# valid_qa_runs = qa_runs.loc[
#     (qa_runs["has_answer_in_corpus"] == 1) &
#     (qa_runs["is_noanswer_probe"] == 0)
# ].copy()

# valid_qa_runs.shape

(3800, 49)

In [9]:
# valid_qa_runs[[
#     "example_id",
#     "query",
#     "doc_ids_used",
#     "chunk_ids_used",
#     "domain",
#     "task_type",
#     "difficulty"
# ]].head()

,example_id,query,doc_ids_used,chunk_ids_used,domain,task_type,difficulty
0,QA000001,Which segment contributed most to revenue growth?,DOC0523|DOC0323|DOC0426|DOC0253,C04183|C02494|C03407|C01959,financial_reports,multi_hop,medium
1,QA000002,Which segment contributed most to revenue growth?,DOC0502|DOC0631|DOC0206|DOC0240,C03969|C05043|C01579|C01864,financial_reports,factoid,hard
2,QA000003,Which segment contributed most to revenue growth?,DOC0587|DOC0371|DOC0533|DOC0323,C04650|C02964|C04250|C02491,financial_reports,explanation,hard
3,QA000004,What is the standard probation period for new ...,DOC0118|DOC0329|DOC0356|DOC0573,C00875|C02553|C02859|C04538,hr_policies,explanation,medium
4,QA000005,Which symptoms indicate the need for immediate...,DOC0500|DOC0391|DOC0509|DOC0442,C03944|C03135|C04062|C03526,medical_guides,multi_hop,hard


In [10]:
# valid_example_ids = set(valid_qa_runs["example_id"])

# retrievals_valid = retrievals[
#     retrievals["example_id"].isin(valid_example_ids)
# ].copy()

# retrievals_valid.shape

(93135, 12)

In [11]:
# retrievals_valid

,run_id,example_id,scenario_id,query_id,split,query_domain,difficulty,retrieval_strategy,rank,chunk_id,retrieval_score,is_relevant
0,run_0,QA000001,SC0014,Q0024,train,financial_reports,medium,hybrid,1,C04183,0.812784,0
1,run_0,QA000001,SC0014,Q0024,train,financial_reports,medium,hybrid,2,C04637,0.342564,0
2,run_0,QA000001,SC0014,Q0024,train,financial_reports,medium,hybrid,3,C01460,0.377480,0
3,run_0,QA000001,SC0014,Q0024,train,financial_reports,medium,hybrid,4,C02494,0.684696,0
4,run_0,QA000001,SC0014,Q0024,train,financial_reports,medium,hybrid,5,C04748,0.491910,0
...,...,...,...,...,...,...,...,...,...,...,...,...
93130,run_3799,QA003800,SC0037,Q0002,train,product_docs,easy,dense_then_rerank,4,C02786,0.708525,1
93131,run_3799,QA003800,SC0037,Q0002,train,product_docs,easy,dense_then_rerank,5,C04366,0.635868,1
93132,run_3799,QA003800,SC0037,Q0002,train,product_docs,easy,dense_then_rerank,6,C03781,0.684503,1
93133,run_3799,QA003800,SC0037,Q0002,train,product_docs,easy,dense_then_rerank,7,C02635,0.533189,1


In [12]:
# retrievals_valid["is_relevant"].value_counts()

is_relevant
0    65395
1    27740
Name: count, dtype: int64

In [13]:
# retrievals_valid["rank"].describe()

count    93135.000000
mean        15.512181
std         10.486536
min          1.000000
25%          7.000000
50%         14.000000
75%         23.000000
max         44.000000
Name: rank, dtype: float64

In [14]:
# from collections import defaultdict

# # example_id -> list of (rank, is_relevant)
# retrievals_by_example = defaultdict(list)

# for _, row in retrievals_valid.iterrows():
#     retrievals_by_example[row["example_id"]].append(
#         (row["rank"], row["is_relevant"])
#     )

# # Sort by rank per query
# for ex_id in retrievals_by_example:
#     retrievals_by_example[ex_id].sort(key=lambda x: x[0])

In [15]:
# def precision_at_k(retrieved, k):
#     """
#     retrieved: list of (rank, is_relevant), sorted by rank
#     """
#     top_k = retrieved[:k]
#     if not top_k:
#         return 0.0

#     relevant_count = sum(rel for _, rel in top_k)
#     return relevant_count / k

In [16]:
# def recall_at_k(retrieved, k):
#     """
#     retrieved: list of (rank, is_relevant), sorted by rank
#     """
#     total_relevant = sum(rel for _, rel in retrieved)
#     if total_relevant == 0:
#         return 0.0

#     top_k = retrieved[:k]
#     relevant_in_top_k = sum(rel for _, rel in top_k)
#     return relevant_in_top_k / total_relevant

In [17]:
# def reciprocal_rank(retrieved):
#     """
#     retrieved: list of (rank, is_relevant), sorted by rank
#     """
#     for rank, rel in retrieved:
#         if rel == 1:
#             return 1.0 / rank
#     return 0.0

In [18]:
# import numpy as np

# Ks = [5, 10]

# metrics = {
#     "precision": {k: [] for k in Ks},
#     "recall": {k: [] for k in Ks},
#     "rr": []
# }

# for ex_id, retrieved in retrievals_by_example.items():
#     for k in Ks:
#         metrics["precision"][k].append(precision_at_k(retrieved, k))
#         metrics["recall"][k].append(recall_at_k(retrieved, k))
#     metrics["rr"].append(reciprocal_rank(retrieved))

In [19]:
# metrics.keys()

dict_keys(['precision', 'recall', 'rr'])

In [20]:
# results = {}

# for k in Ks:
#     results[f"Precision@{k}"] = np.mean(metrics["precision"][k])
#     results[f"Recall@{k}"] = np.mean(metrics["recall"][k])

# results["MRR"] = np.mean(metrics["rr"])

# results

{'Precision@5': 0.7049473684210527,
 'Recall@5': 0.4364899464570517,
 'Precision@10': 0.7080000000000001,
 'Recall@10': 0.9008832972582973,
 'MRR': 0.7825669900497467}

In [21]:
# valid_example_difficulty = (
#     valid_qa_runs
#     .set_index("example_id")["difficulty"]
# )

# difficulty_metrics = defaultdict(list)

# for ex_id, retrieved in retrievals_by_example.items():
#     diff = valid_example_difficulty.loc[ex_id]
#     difficulty_metrics[diff].append(reciprocal_rank(retrieved))

# {diff: np.mean(vals) for diff, vals in difficulty_metrics.items()}

{'medium': 0.7853215618412493,
 'hard': 0.6517022326591264,
 'easy': 0.8835279101094199}

In [22]:
# Re-rankinf evaluationm

In [23]:
# from src.retrieval.bm25 import BM25Index
# from src.retrieval.faiss import FAISSIndex
# from src.storage.metadata import MetadataStore
# from src.retrieval.hybridretriver import HybridRetriever
# from src.security.user_context import UserContext

In [24]:
import importlib
import src.retrieval.hybridretriver as hybrid_retriever
import src.reranking.crossencoder as cross_encoder
importlib.reload(hybrid_retriever)
importlib.reload(cross_encoder)

<module 'src.reranking.crossencoder' from 'C:\\Users\\Vohita\\RAG\\src\\reranking\\crossencoder.py'>

In [25]:
EVAL_SAMPLE_SIZE = 200

eval_subset = valid_qa_runs.sample(
    n=EVAL_SAMPLE_SIZE,
    random_state=42
)

In [26]:
def run_retrieval(retriever, query, user, top_k=5):
    query_embedding = embedder.encode(
        query,
        convert_to_numpy=True
    ).reshape(1, -1)

    return retriever.retrieve(
        query=query,
        query_embedding=query_embedding,
        user=user,
        top_k=top_k
    )

In [27]:
ground_truth_chunks = (
    valid_qa_runs
    .set_index("example_id")["chunk_ids_used"]
)

In [28]:
import ast
import re

def parse_chunk_ids(chunk_ids_value):
    """
    Robust parser for chunk_ids_used column.

    Returns:
        set of chunk_id strings
    """
    if pd.isna(chunk_ids_value):
        return set()

    # Case 1: already a list-like string
    try:
        parsed = ast.literal_eval(chunk_ids_value)
        if isinstance(parsed, list):
            return set(parsed)
        if isinstance(parsed, str):
            return {parsed}
    except Exception:
        pass

    # Case 2: fallback — extract chunk ids via regex
    return set(re.findall(r"chunk_[a-zA-Z0-9_]+", str(chunk_ids_value)))

In [29]:
def compute_mrr_from_results(results, relevant_chunks):
    for rank, (row_id, _) in enumerate(results, start=1):
        chunk_id = chunks.iloc[row_id]["chunk_id"]
        if chunk_id in relevant_chunks:
            return 1.0 / rank
    return 0.0

In [30]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [31]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

In [32]:
from src.retrieval.bm25 import BM25Index
bm25 = BM25Index(f"{DATA_PATH}/indexes/bm25.pkl")


faiss_index = FAISSIndex(f"{DATA_PATH}/indexes/faiss.index")
faiss_index.load()

metadata_df = pd.read_csv(f"{DATA_PATH}/metadata/chunk_metadata.csv")
metadata_store = MetadataStore(metadata_df)

In [33]:
reranker = cross_encoder.CrossEncoderReranker()

In [34]:
retriever_no_rerank = hybrid_retriever.HybridRetriever(
    bm25=bm25,
    faiss=faiss_index,
    metadata_store=metadata_store,
    reranker=None,
    chunks_df=chunks
)

In [35]:
retriever_with_rerank = hybrid_retriever.HybridRetriever(
    bm25=bm25,
    faiss=faiss_index,
    metadata_store=metadata_store,
    chunks_df=chunks,          # <-- ADD THIS
    reranker=reranker
)

In [36]:
from tqdm import tqdm

baseline_rr = []
reranked_rr = []

user = UserContext(
    user_id="eval_user",
    department="engineering",
    clearance=2,
    projects=[]
)

for _, row in tqdm(eval_subset.iterrows(), total=len(eval_subset)):
    query = row["query"]
    example_id = row["example_id"]
    relevant_chunks = parse_chunk_ids(
        ground_truth_chunks.loc[example_id]
    )

    # Baseline
    base_results = run_retrieval(
        retriever_no_rerank,
        query,
        user
    )
    baseline_rr.append(
        compute_mrr_from_results(base_results, relevant_chunks)
    )

    # With reranker
    rerank_results = run_retrieval(
        retriever_with_rerank,
        query,
        user
    )
    reranked_rr.append(
        compute_mrr_from_results(rerank_results, relevant_chunks)
    )

100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [00:43<00:00,  4.62it/s]


In [37]:
import numpy as np

print("Baseline MRR:", np.mean(baseline_rr))
print("Reranked MRR:", np.mean(reranked_rr))
print("Delta:", np.mean(reranked_rr) - np.mean(baseline_rr))

Baseline MRR: 0.0
Reranked MRR: 0.0
Delta: 0.0


In [38]:
improvements = np.array(reranked_rr) - np.array(baseline_rr)

print("Improved queries:", np.sum(improvements > 0))
print("Regressed queries:", np.sum(improvements < 0))
print("Unchanged:", np.sum(improvements == 0))

Improved queries: 0
Regressed queries: 0
Unchanged: 200


In [39]:
difficulty_map = eval_subset.set_index("example_id")["difficulty"]

by_difficulty = {}

for diff in ["easy", "medium", "hard"]:
    idxs = [
        i for i, ex_id in enumerate(eval_subset["example_id"])
        if difficulty_map.loc[ex_id] == diff
    ]
    by_difficulty[diff] = np.mean(
        np.array(reranked_rr)[idxs] - np.array(baseline_rr)[idxs]
    )

by_difficulty

{'easy': 0.0, 'medium': 0.0, 'hard': 0.0}

In [40]:
regressed_idxs = np.where(improvements < 0)[0][:5]

for i in regressed_idxs:
    print("="*80)
    print("QUERY:", eval_subset.iloc[i]["query"])
    print("Baseline:", baseline_rr[i])
    print("Reranked:", reranked_rr[i])

    print("\nBaseline result:")
    print(chunks.iloc[
        run_retrieval(retriever_no_rerank, eval_subset.iloc[i]["query"], user)[0][0]
    ]["chunk_text"][:300])

    print("\nReranked result:")
    print(chunks.iloc[
        run_retrieval(retriever_with_rerank, eval_subset.iloc[i]["query"], user)[0][0]
    ]["chunk_text"][:300])

In [41]:
# Rank movement analysis: Did reranking move candidates up/down?

In [42]:
def extract_row_ids(results):
    return [row_id for row_id, _ in results]

In [43]:
# rank_changes = []

# for _, row in eval_subset.iterrows():
#     query = row["query"]

#     base_results = run_retrieval(
#         retriever_no_rerank, query, user
#     )
#     rerank_results = run_retrieval(
#         retriever_with_rerank, query, user
#     )

#     base_order = extract_row_ids(base_results)
#     rerank_order = extract_row_ids(rerank_results)

#     if base_order != rerank_order:
#         rank_changes.append(1)
#     else:
#         rank_changes.append(0)

In [44]:
# change_rate = sum(rank_changes) / len(rank_changes)
# change_rate

1.0

In [45]:
# Semantic similarity to query: Are top results more semantically aligned after reranking?
# from sklearn.metrics.pairwise import cosine_similarity

In [46]:
# def avg_query_similarity(query_emb, row_ids, chunk_embeddings):
#     sims = []
#     for row_id in row_ids:
#         sims.append(
#             cosine_similarity(
#                 query_emb,
#                 chunk_embeddings[row_id].reshape(1, -1)
#             )[0][0]
#         )
#     return float(np.mean(sims))

In [47]:
# import numpy as np

# embeddings = np.load("../data/embeddings.npy")
# embeddings.shape

(5237, 384)

In [48]:
baseline_sims = []
reranked_sims = []

for _, row in eval_subset.iterrows():
    query = row["query"]
    query_emb = embedder.encode(
        query, convert_to_numpy=True
    ).reshape(1, -1)

    base_results = run_retrieval(
        retriever_no_rerank, query, user
    )
    rerank_results = run_retrieval(
        retriever_with_rerank, query, user
    )

    baseline_sims.append(
        avg_query_similarity(
            query_emb,
            extract_row_ids(base_results),
            embeddings
        )
    )

    reranked_sims.append(
        avg_query_similarity(
            query_emb,
            extract_row_ids(rerank_results),
            embeddings
        )
    )

In [49]:
np.mean(baseline_sims), np.mean(reranked_sims)

(0.22366179056465627, 0.37655811741948125)

In [50]:
baseline_mean = np.mean(baseline_sims)
reranked_mean = np.mean(reranked_sims)

baseline_mean, reranked_mean


(0.22366179056465627, 0.37655811741948125)

In [51]:
print("Base results:", base_results)
print("Rerank results:", rerank_results)


Base results: [(25, 1.0), (3430, 1.0), (12, 0.5), (3432, 0.5), (2507, 0.3333333333333333)]
Rerank results: [(2502, -4.472016334533691), (2507, -5.086071968078613), (3430, -5.167309284210205), (3431, -5.312104225158691), (3432, -5.397626876831055)]


In [52]:
base_row_ids = extract_row_ids(base_results)
rerank_row_ids = extract_row_ids(rerank_results)

print("Base row ids:", base_row_ids[:5])
print("Rerank row ids:", rerank_row_ids[:5])
print("Max embedding index:", embeddings.shape[0] - 1)




Base row ids: [25, 3430, 12, 3432, 2507]
Rerank row ids: [2502, 2507, 3430, 3431, 3432]
Max embedding index: 5236


In [53]:
def avg_query_similarity(query_emb, row_ids, embeddings):
    if len(row_ids) == 0:
        return np.nan

    sims = []
    for rid in row_ids:
        if rid >= embeddings.shape[0]:
            continue

        doc_emb = embeddings[rid].reshape(1, -1)
        sim = np.dot(query_emb, doc_emb.T) / (
            np.linalg.norm(query_emb) * np.linalg.norm(doc_emb)
        )
        sims.append(sim.item())

    if len(sims) == 0:
        return np.nan

    return np.mean(sims)


In [54]:
baseline_sims = []
reranked_sims = []

for _, row in eval_subset.iterrows():
    query = row["query"]
    query_emb = embedder.encode(query, convert_to_numpy=True).reshape(1, -1)

    base_results = run_retrieval(retriever_no_rerank, query, user)
    rerank_results = run_retrieval(retriever_with_rerank, query, user)

    baseline_sims.append(
        avg_query_similarity(
            query_emb,
            extract_row_ids(base_results),
            embeddings
        )
    )

    reranked_sims.append(
        avg_query_similarity(
            query_emb,
            extract_row_ids(rerank_results),
            embeddings
        )
    )


In [55]:
baseline_sims = np.array(baseline_sims)
reranked_sims = np.array(reranked_sims)

baseline_mean = np.nanmean(baseline_sims)
reranked_mean = np.nanmean(reranked_sims)

baseline_mean, reranked_mean


(0.22366178224934266, 0.376558122315444)

In [56]:
type(baseline_sims), type(reranked_sims)


(numpy.ndarray, numpy.ndarray)

In [57]:
deltas = reranked_sims - baseline_sims
np.percentile(deltas, [10, 25, 50, 75, 90])


array([-0.00555689,  0.0046802 ,  0.15451155,  0.28301089,  0.29368376])

In [58]:
baseline_sims[:10], reranked_sims[:10]


(array([0.32381802, 0.19747031, 0.20227026, 0.20227026, 0.21599402,
        0.3009921 , 0.16815048, 0.20227026, 0.17995461, 0.247526  ]),
 array([0.47832957, 0.62554578, 0.19671338, 0.19671338, 0.4412919 ,
        0.46697737, 0.17283068, 0.19671338, 0.16626949, 0.26554296]))

In [59]:
np.isnan(baseline_sims).mean(), np.isnan(reranked_sims).mean()


(0.0, 0.0)

In [60]:
np.nanpercentile(deltas, [10, 25, 50, 75, 90])


array([-0.00555689,  0.0046802 ,  0.15451155,  0.28301089,  0.29368376])

In [61]:
np.mean(deltas > 0)


0.765

In [62]:
baseline_sims = np.array(baseline_sims, dtype=float)
reranked_sims = np.array(reranked_sims, dtype=float)


In [63]:
np.nanmean(reranked_sims - baseline_sims)


0.15289634006610137

In [65]:
np.mean(baseline_sims), np.mean(reranked_sims)

(0.22366178224934266, 0.376558122315444)

In [66]:
np.mean(np.array(reranked_sims) - np.array(baseline_sims))

0.15289634006610137

In [67]:
delta = reranked_sims - baseline_sims

print("Baseline mean:", np.nanmean(baseline_sims))
print("Reranked mean:", np.nanmean(reranked_sims))
print("Mean improvement:", np.nanmean(delta))
print("Win rate:", np.nanmean(delta > 0))
print("Percentiles:", np.nanpercentile(delta, [10, 25, 50, 75, 90]))


Baseline mean: 0.22366178224934266
Reranked mean: 0.376558122315444
Mean improvement: 0.15289634006610137
Win rate: 0.765
Percentiles: [-0.00555689  0.0046802   0.15451155  0.28301089  0.29368376]


In [ ]:
# Qualitative inspection: Does the top-1/top-3 look better to a human?

In [68]:
changed_idxs = [
    i for i, v in enumerate(rank_changes) if v == 1
][:5]

In [69]:
for i in changed_idxs:
    query = eval_subset.iloc[i]["query"]

    print("="*80)
    print("QUERY:", query)

    print("\nBASELINE TOP-1:")
    base = run_retrieval(
        retriever_no_rerank, query, user
    )
    print(chunks.iloc[base[0][0]]["chunk_text"][:400])

    print("\nRERANKED TOP-1:")
    rerank = run_retrieval(
        retriever_with_rerank, query, user
    )
    print(chunks.iloc[rerank[0][0]]["chunk_text"][:400])

QUERY: How is user consent stored for analytics tracking?

BASELINE TOP-1:
You are looking at part of a written policy that employees or customers might need to consult. In this section of 'Data and access policy — 2020 update' sets out the formal policy language that staff must follow. It defines how privacy and terms are interpreted in audits, which teams are accountable, and how violations are reported and remediated within agreed service levels.

RERANKED TOP-1:
You are looking at part of a written policy that employees or customers might need to consult. In this section of 'Data and access policy — 2020 update' sets out the formal policy language that staff must follow. It defines how privacy and terms are interpreted in audits, which teams are accountable, and how violations are reported and remediated within agreed service levels.
QUERY: Which segment contributed most to revenue growth?

BASELINE TOP-1:
The following text comes from a support FAQ article used in a help center. I